# 🏛️ Итоговое домашнее задание
# «Аналитик реестра — Обработка данных ЕФРСБ»





Вы — юрист, анализирующий данные из **Единого федерального реестра сведений о банкротстве (ЕФРСБ)**.
Вам поступила выгрузка в формате JSON с информацией о сообщениях по делам о банкротстве за 2025 год.

**Ваша задача** — извлечь ключевые данные, очистить от мусора, сопоставить с реестром организаций
и сформировать аналитический отчёт для руководства.

---
### 📋 Как выполнять и сдавать задание

**Где выполнять:**
- **Вариант 1 (рекомендуется):** Загрузите файл `.ipynb` в [Google Colab](https://colab.research.google.com/), загрузите папку `final_hw_data/` в среду выполнения (например, через боковую панель «Файлы» или смонтировав Google Drive).
- **Вариант 2:** Работайте локально в Jupyter Notebook / JupyterLab. Убедитесь, что папка `final_hw_data/` находится в той же директории, что и файл задания.

**Как сдавать на проверку:**
- Отправьте **ссылку на Google Colab** с общим доступом («Любой, у кого есть ссылка — комментатор»), **ИЛИ**
- Отправьте **ссылку на GitHub-репозиторий**, в котором лежит выполненный `.ipynb`-файл и папка `final_hw_data/` с результатами.

> ⚠️ Убедитесь, что ссылка открывается в режиме просмотра и все ячейки выполнены (результаты видны).

---

### 📦 Исходные данные

В папке `final_hw_data/` находятся 3 файла:

| Файл | Описание |
|------|----------|
| `bankruptcy_messages.json` | Сообщения ЕФРСБ |
| `organizations.json` | Реестр организаций |
| `priority_cases.txt` | Номера приоритетных дел |



---
# 🟢 Загрузка и кросс-референс (0-2 балла)
---

### Задание 1.1 — Загрузка данных из файлов

Загрузите данные из трёх файлов:
- `bankruptcy_messages.json` → список `messages`
- `organizations.json` → список `organizations`
- `priority_cases.txt` → список `priority_case_numbers`

Выведите количество загруженных записей из каждого источника.

In [23]:
import json
from pathlib import Path
path_messages = Path("/content/final_hw_data/bankruptcy_messages.json")
path_organizations = Path("/content/final_hw_data/organizations.json")
path_priority_cases = Path("/content/final_hw_data/priority_cases.txt")
save_dir = Path("/content/final_hw_data")

In [24]:
# Загрузка данных из bankruptcy_messages.json
with open(path_messages, 'r', encoding='utf-8') as f:
    messages = json.load(f)

# Загрузка данных из organizations.json

with open(path_organizations, 'r', encoding='utf-8') as f:
    organizations = json.load(f)

# Загрузка данных из priority_cases.txt
with open(path_priority_cases, 'r', encoding='utf-8') as f:
    priority_cases = [line.strip() for line in f if line.strip()]

# Вывод количества записей
print(f"Количество сообщений о банкротстве: {len(messages)}")
print(f"Количество организаций: {len(organizations)}")
print(f"Количество приоритетных дел: {len(priority_cases)}")

Количество сообщений о банкротстве: 54
Количество организаций: 30
Количество приоритетных дел: 10


### Задание 1.2 — Словарь организаций

Создайте словарь `org_by_inn`, где ключ — ИНН (строка), а значение — словарь с данными об организации.
Используйте **dict comprehension**.

Выведите количество организаций в словаре и информацию по ИНН `"7701234567"`.

In [25]:
org_by_inn = {org['inn']: org for org in organizations}

# Вывод количества организаций в словаре
print(f"Количество организаций в словаре: {len(org_by_inn)}")

# Вывод информации по ИНН "7701234567"
print(f"\nИнформация по ИНН 7701234567:")
print(org_by_inn.get("7701234567", "Организация не найдена"))

Количество организаций в словаре: 30

Информация по ИНН 7701234567:
{'inn': '7701234567', 'ogrn': '1027700123456', 'name': 'ООО "Альфа-Строй"', 'address': 'г. Москва, ул. Строителей, д. 15', 'region': 'Москва'}


### Задание 1.3 — Кросс-референс: связываем сообщения с организациями

Для каждого сообщения добавьте поле `org_name` (название организации) и `region`,
используя словарь `org_by_inn` и метод `.get()`.
Если организация не найдена — ставьте значение `None`.

Посчитайте, сколько сообщений удалось связать с организацией,
а сколько — не удалось.

In [26]:
for msg in messages:
    publisher_inn = msg.get('publisher_inn', '')
    org_data = org_by_inn.get(publisher_inn)

    if org_data:
        msg['org_name'] = org_data['name']
        msg['region'] = org_data['region']
    else:
        msg['org_name'] = None
        msg['region'] = None

# Подсчет количества сообщений
linked_count = sum(1 for msg in messages if msg['org_name'] is not None)
not_linked_count = len(messages) - linked_count

print(f"Всего сообщений: {len(messages)}")
print(f"Удалось связать с организацией: {linked_count}")
print(f"Не удалось связать: {not_linked_count}")

Всего сообщений: 54
Удалось связать с организацией: 51
Не удалось связать: 3


### Задание 1.4 — Множества и приоритетные дела

1. Создайте множество `priority_set` из списка `priority_case_numbers`.
2. Создайте множество `message_case_set` из номеров дел в `messages`
   (используйте list comprehension + filter для непустых номеров).
3. Найдите пересечение — номера дел, которые есть и в приоритетных, и в сообщениях (`&`).
4. Выведите результат.

In [27]:
priority_set = set(priority_cases)

# Создание множества из номеров дел (только непустые)
message_case_set = {msg['case_number'] for msg in messages if msg.get('case_number') and msg['case_number'].strip()}

# Нахождение пересечения
intersection = priority_set & message_case_set

print(f"Количество приоритетных дел: {len(priority_set)}")
print(f"Количество уникальных номеров дел в сообщениях: {len(message_case_set)}")
print(f"\nПересечение (дела в приоритетном списке и в сообщениях):")
print(f"Количество: {len(intersection)}")
print(f"Номера дел: {sorted(intersection)}")

Количество приоритетных дел: 10
Количество уникальных номеров дел в сообщениях: 16

Пересечение (дела в приоритетном списке и в сообщениях):
Количество: 10
Номера дел: ['А60-11111/2025', 'А60-12345/2025', 'А60-22222/2025', 'А60-33333/2025', 'А60-44444/2025', 'А60-56789/2025', 'А60-66666/2025', 'А60-77777/2025', 'А60-88888/2025', 'А60-99999/2025']


---
# 🟡 Очистка и валидация (0-3 балла)
---

### Задание 2.1 — Функция парсинга дат

Напишите функцию `parse_date(date_str)`, которая принимает строку с датой
и возвращает объект `datetime` или `None`, если распарсить не удалось.

Форматы для обработки:
- `"DD.MM.YYYY"` — `strptime`
- `"YYYY-MM-DD"` — `fromisoformat`
- `"DD месяца YYYY г."` — замена русских месяцев + `strptime`
- `"DD/MM/YYYY HH:MM"` — `strptime`

Обрабатывайте `None` и пустые строки через `try/except`.

In [28]:
def parse_date(date_str):
    """
    Парсит строку с датой в различных форматах.
    Возвращает объект datetime или None, если распарсить не удалось.
    """
    from datetime import datetime

    # Обработка None и пустых строк
    if date_str is None or date_str == "":
        return None

    # Убираем лишние пробелы
    date_str = str(date_str).strip()

    # Словарь для замены русских названий месяцев
    months_ru = {
        'января': '01', 'февраля': '02', 'марта': '03', 'апреля': '04',
        'мая': '05', 'июня': '06', 'июля': '07', 'августа': '08',
        'сентября': '09', 'октября': '10', 'ноября': '11', 'декабря': '12'
    }

    try:
        # Формат 1: "DD.MM.YYYY"
        if '.' in date_str and date_str.count('.') == 2:
            # Пробуем сначала полный формат с годом из 4 цифр
            try:
                return datetime.strptime(date_str, "%d.%m.%Y")
            except ValueError:
                # Если не получилось, пробуем с годом из 2 цифр
                return datetime.strptime(date_str, "%d.%m.%y")

        # Формат 2: "YYYY-MM-DD" (ISO формат)
        elif '-' in date_str and date_str.count('-') == 2:
            return datetime.fromisoformat(date_str)

        # Формат 3: "DD месяца YYYY г." (например: "20 марта 2025 г.")
        elif ' ' in date_str and 'г.' in date_str:
            for month_ru, month_num in months_ru.items():
                if month_ru in date_str:
                    # Заменяем русское название месяца на числовое
                    date_str_fixed = date_str.replace(month_ru, month_num)
                    # Убираем "г." и лишние пробелы
                    date_str_fixed = date_str_fixed.replace(' г.', '').strip()
                    return datetime.strptime(date_str_fixed, "%d %m %Y")

        # Формат 4: "DD/MM/YYYY HH:MM"
        elif '/' in date_str and ' ' in date_str and ':' in date_str:
            return datetime.strptime(date_str, "%d/%m/%Y %H:%M")

        # Формат 5: "DD/MM/YYYY" (без времени)
        elif '/' in date_str and date_str.count('/') == 2:
            return datetime.strptime(date_str, "%d/%m/%Y")

        # Если ничего не подошло
        return None

    except (ValueError, TypeError, AttributeError):
        return None

# Тестирование функции на примерах из файла
print("Тестирование функции parse_date (с использованием datetime):")
print("-" * 50)

test_dates = [
    "15.01.2025",
    "2025-02-10",
    "20 марта 2025 г.",
    "15/03/2025 10:30",
    None,
    "",
    "ДАТА УТЕРЯНА",
    "10.02.2025"
]

for test_date in test_dates:
    result = parse_date(test_date)
    if result:
        print(f"'{test_date}' -> {result.strftime('%Y-%m-%d %H:%M:%S') if result else result}")
    else:
        print(f"'{test_date}' -> {result}")

Тестирование функции parse_date (с использованием datetime):
--------------------------------------------------
'15.01.2025' -> 2025-01-15 00:00:00
'2025-02-10' -> 2025-02-10 00:00:00
'20 марта 2025 г.' -> 2025-03-20 00:00:00
'15/03/2025 10:30' -> 2025-03-15 10:30:00
'None' -> None
'' -> None
'ДАТА УТЕРЯНА' -> None
'10.02.2025' -> 2025-02-10 00:00:00


### Задание 2.2 — Функция валидации сообщения

Напишите функцию `validate_message(msg)`, которая:

1. Проверяет наличие **обязательных полей**: `publisher_inn`, `msg_text`, `date_published`, `type`, `case_number`.
   Если поле отсутствует — ошибка типа `KeyError`.
2. Проверяет, что **ИНН** состоит из 10 или 12 цифр (метод `.isdigit()`).
3. Парсит дату через `parse_date()`. Если дата не парсится — ошибка.
4. Проверяет, что **тип сообщения** — непустая строка.

Функция возвращает кортеж `(valid_msg, errors)`:
- `valid_msg` — словарь с очищенными данными (или `None`)
- `errors` — список строк с описанием ошибок

In [29]:
def validate_message(msg):
    """
    Проверяет сообщение на корректность данных.
    Возвращает кортеж (valid_msg, errors):
    - valid_msg: словарь с очищенными данными или None
    - errors: список строк с описанием ошибок
    """
    errors = []
    required_fields = ['publisher_inn', 'msg_text', 'date_published', 'type', 'case_number']

    # Проверка наличия обязательных полей
    for field in required_fields:
        if field not in msg:
            errors.append(f"Отсутствует обязательное поле: '{field}'")
        elif msg[field] is None:
            errors.append(f"Поле '{field}' имеет значение None")

    # Если есть ошибки в обязательных полях, возвращаем None
    if errors:
        return (None, errors)

    # Проверка ИНН (10 или 12 цифр)
    inn = str(msg['publisher_inn']).strip()
    if not inn.isdigit() or (len(inn) != 10 and len(inn) != 12):
        errors.append(f"Неверный формат ИНН: '{inn}' (должен состоять из 10 или 12 цифр)")

    # Парсинг даты
    date_published = msg['date_published']
    parsed_date = parse_date(date_published)
    if parsed_date is None:
        errors.append(f"Не удалось распарсить дату: '{date_published}'")

    # Проверка типа сообщения
    msg_type = msg['type']
    if msg_type is None or str(msg_type).strip() == "":
        errors.append("Тип сообщения не может быть пустым")

    # Проверка текста сообщения
    msg_text = msg['msg_text']
    if msg_text is None or str(msg_text).strip() == "":
        errors.append("Текст сообщения не может быть пустым")

    # Проверка номера дела
    case_number = msg['case_number']
    if case_number is None or str(case_number).strip() == "":
        errors.append("Номер дела не может быть пустым")

    # Если есть ошибки, возвращаем None
    if errors:
        return (None, errors)

    # Создаем очищенный словарь с данными
    valid_msg = {
        'publisher_inn': inn,
        'msg_text': str(msg_text).strip(),
        'date_published': parsed_date,
        'type': str(msg_type).strip(),
        'case_number': str(case_number).strip()
    }

    # Добавляем дополнительные поля, если они есть
    if 'org_name' in msg and msg['org_name'] is not None:
        valid_msg['org_name'] = msg['org_name']
    if 'region' in msg and msg['region'] is not None:
        valid_msg['region'] = msg['region']

    return (valid_msg, errors)

# Тестирование функции на нескольких сообщениях
print("Тестирование функции validate_message:")
print("-" * 60)

# Тестируем первые 5 сообщений
for i, msg in enumerate(messages[:5]):
    print(f"\nСообщение {i+1}:")
    print(f"  ИНН: {msg.get('publisher_inn')}")
    print(f"  Дата: {msg.get('date_published')}")
    print(f"  Тип: {msg.get('type')}")

    valid_msg, errors = validate_message(msg)

    if valid_msg:
        print(f"  Результат: ✅ Верно")
        print(f"  Очищенная дата: {valid_msg['date_published'].strftime('%Y-%m-%d')}")
    else:
        print(f"  Результат: ❌ Ошибка")
        print(f"  Ошибки:")
        for error in errors:
            print(f"    - {error}")

# Подсчет статистики по всем сообщениям
print("\n" + "=" * 60)
print("СТАТИСТИКА ПО ВСЕМ СООБЩЕНИЯМ:")
print("-" * 60)

valid_count = 0
invalid_count = 0
all_errors = []

for msg in messages:
    valid_msg, errors = validate_message(msg)
    if valid_msg:
        valid_count += 1
    else:
        invalid_count += 1
        all_errors.extend(errors)

print(f"Валидных сообщений: {valid_count}")
print(f"Невалидных сообщений: {invalid_count}")
print(f"Всего сообщений: {len(messages)}")

# Выводим уникальные типы ошибок
if all_errors:
    print("\nУникальные типы ошибок:")
    unique_errors = set(all_errors)
    for error in sorted(unique_errors):
        count = all_errors.count(error)
        print(f"  - {error} (встречается {count} раз)")

Тестирование функции validate_message:
------------------------------------------------------------

Сообщение 1:
  ИНН: 7701234567
  Дата: 15.01.2025
  Тип: Введение процедуры
  Результат: ✅ Верно
  Очищенная дата: 2025-01-15

Сообщение 2:
  ИНН: 7702345678
  Дата: 2025-02-10
  Тип: Собрание кредиторов
  Результат: ✅ Верно
  Очищенная дата: 2025-02-10

Сообщение 3:
  ИНН: 6658123450
  Дата: 2025-03-15T00:00:00
  Тип: Завершение процедуры
  Результат: ✅ Верно
  Очищенная дата: 2025-03-15

Сообщение 4:
  ИНН: 7810345612
  Дата: 20 марта 2025 г.
  Тип: Признание банкротом
  Результат: ✅ Верно
  Очищенная дата: 2025-03-20

Сообщение 5:
  ИНН: 5027123456
  Дата: 2025-04-05
  Тип: Продажа имущества
  Результат: ✅ Верно
  Очищенная дата: 2025-04-05

СТАТИСТИКА ПО ВСЕМ СООБЩЕНИЯМ:
------------------------------------------------------------
Валидных сообщений: 48
Невалидных сообщений: 6
Всего сообщений: 54

Уникальные типы ошибок:
  - Не удалось распарсить дату: 'ДАТА УТЕРЯНА' (встречается 1 

### Задание 2.3 — Массовая валидация

Примените `validate_message()` ко всем сообщениям.
Разделите результат на два списка: `valid_messages` и `error_records`.

Соберите **статистику ошибок** по типам (сколько `KeyError`, `ValueError` и т.д.).

In [30]:
from collections import defaultdict
valid_messages = []
error_records = []
error_stats = defaultdict(int)

# Применяем валидацию ко всем сообщениям
for idx, msg in enumerate(messages):
    valid_msg, errors = validate_message(msg)

    if valid_msg:
        valid_messages.append(valid_msg)
    else:
        error_records.append({
            'index': idx,
            'original_message': msg,
            'errors': errors
        })
        # Собираем статистику по типам ошибок
        for error in errors:
            # Извлекаем тип ошибки (KeyError или ValueError)
            if error.startswith("KeyError"):
                error_stats["KeyError"] += 1
            elif error.startswith("ValueError"):
                error_stats["ValueError"] += 1
            else:
                error_stats["Other"] += 1

# Вывод результатов
print(f"\nРезультаты валидации:")
print(f"  ✅ Валидных сообщений: {len(valid_messages)}")
print(f"  ❌ Невалидных сообщений: {len(error_records)}")
print(f"  📊 Всего сообщений: {len(messages)}")

# Статистика по типам ошибок
print(f"\nСтатистика ошибок по типам:")
for error_type, count in sorted(error_stats.items()):
    print(f"  {error_type}: {count}")

# Детальный анализ ошибок
print(f"\nДетальный анализ ошибок:")
detailed_errors = defaultdict(int)

for error_record in error_records:
    for error in error_record['errors']:
        detailed_errors[error] += 1

print("\nНаиболее частые ошибки:")
for error, count in sorted(detailed_errors.items(), key=lambda x: x[1], reverse=True)[:5]:
    print(f"  • {error} (встречается {count} раз)")

# Вывод примеров невалидных сообщений
if error_records:
    print(f"\nПримеры невалидных сообщений (первые 3):")
    for i, error_record in enumerate(error_records[:3]):
        print(f"\n  Пример {i+1}:")
        print(f"    Индекс сообщения: {error_record['index']}")
        print(f"    ИНН: {error_record['original_message'].get('publisher_inn')}")
        print(f"    Дата: {error_record['original_message'].get('date_published')}")
        print(f"    Тип: {error_record['original_message'].get('type')}")
        print(f"    Ошибки:")
        for error in error_record['errors'][:3]:  # Показываем первые 3 ошибки
            print(f"      - {error}")

# Сохраняем результаты в переменные для дальнейшего использования
print(f"\n✅ Данные сохранены в переменные:")
print(f"  - valid_messages: список из {len(valid_messages)} валидных сообщений")
print(f"  - error_records: список из {len(error_records)} записей с ошибками")
print(f"  - error_stats: словарь со статистикой ошибок")


Результаты валидации:
  ✅ Валидных сообщений: 48
  ❌ Невалидных сообщений: 6
  📊 Всего сообщений: 54

Статистика ошибок по типам:
  Other: 6

Детальный анализ ошибок:

Наиболее частые ошибки:
  • Неверный формат ИНН: '' (должен состоять из 10 или 12 цифр) (встречается 1 раз)
  • Поле 'date_published' имеет значение None (встречается 1 раз)
  • Поле 'type' имеет значение None (встречается 1 раз)
  • Неверный формат ИНН: '7705' (должен состоять из 10 или 12 цифр) (встречается 1 раз)
  • Не удалось распарсить дату: 'ДАТА УТЕРЯНА' (встречается 1 раз)

Примеры невалидных сообщений (первые 3):

  Пример 1:
    Индекс сообщения: 47
    ИНН: 
    Дата: 2025-04-25
    Тип: Подача заявления
    Ошибки:
      - Неверный формат ИНН: '' (должен состоять из 10 или 12 цифр)

  Пример 2:
    Индекс сообщения: 49
    ИНН: 7701234567
    Дата: None
    Тип: Введение процедуры
    Ошибки:
      - Поле 'date_published' имеет значение None

  Пример 3:
    Индекс сообщения: 50
    ИНН: 7702345678
    Дата

---
# 🔵 Извлечение данных и аналитика (0-2 балла)
---

### Задание 3.1 — Извлечение сумм из текста

Напишите функцию `extract_amounts(text)`, которая ищет упоминания денежных сумм в тексте сообщения.

Используйте строковые методы.

Ищите по ключевым словам: `"руб."`, `"тыс. руб."`, `"млн руб."`.

Функция возвращает список строк — найденных сумм.

In [31]:
def extract_amounts(text):
    """
    Ищет упоминания денежных сумм в тексте сообщения.
    Возвращает список строк с найденными суммами.
    """
    if text is None or text == "":
        return []

    text = str(text)
    found_amounts = []

    # Разбиваем текст на слова для удобного поиска
    words = text.split()

    for i, word in enumerate(words):
        # Ищем сумму в рублях
        if 'руб' in word.lower():
            # Смотрим предыдущее слово (там обычно число)
            if i > 0:
                prev_word = words[i-1]
                # Очищаем от запятых и пробелов
                cleaned_prev = prev_word.replace(',', '').strip()

                # Проверяем, является ли предыдущее слово числом
                if cleaned_prev.isdigit():
                    found_amounts.append(f"{cleaned_prev} руб.")
                # Число может быть с пробелом-разделителем тысяч (например "15 000")
                elif ' ' in prev_word:
                    # Убираем пробелы и проверяем
                    no_spaces = prev_word.replace(' ', '')
                    if no_spaces.isdigit():
                        found_amounts.append(f"{no_spaces} руб.")

        # Ищем сумму в тысячах рублей (тыс. руб.)
        if 'тыс.' in word.lower() and i + 1 < len(words) and 'руб' in words[i+1].lower():
            if i > 0:
                prev_word = words[i-1]
                cleaned_prev = prev_word.replace(',', '').strip()

                if cleaned_prev.isdigit():
                    found_amounts.append(f"{cleaned_prev} тыс. руб.")
                elif ' ' in prev_word:
                    no_spaces = prev_word.replace(' ', '')
                    if no_spaces.isdigit():
                        found_amounts.append(f"{no_spaces} тыс. руб.")

        # Ищем сумму в миллионах рублей (млн руб.)
        if 'млн' in word.lower() and i + 1 < len(words) and 'руб' in words[i+1].lower():
            if i > 0:
                prev_word = words[i-1]
                cleaned_prev = prev_word.replace(',', '').strip()

                if cleaned_prev.isdigit():
                    found_amounts.append(f"{cleaned_prev} млн руб.")
                elif ' ' in prev_word:
                    no_spaces = prev_word.replace(' ', '')
                    if no_spaces.isdigit():
                        found_amounts.append(f"{no_spaces} млн руб.")

    # Также ищем суммы в формате "ЧИСЛО руб." без разделителя
    import re
    # Паттерн для чисел с возможными пробелами между цифрами
    pattern = r'(\d{1,3}(?:\s?\d{3})*)\s+руб\.'
    matches = re.findall(pattern, text)
    for match in matches:
        # Убираем пробелы из числа
        clean_number = match.replace(' ', '')
        if clean_number.isdigit():
            amount = f"{clean_number} руб."
            if amount not in found_amounts:
                found_amounts.append(amount)

    # Паттерн для тысяч рублей
    pattern_thousands = r'(\d{1,3}(?:\s?\d{3})*)\s+тыс\.\s+руб\.'
    matches_thousands = re.findall(pattern_thousands, text)
    for match in matches_thousands:
        clean_number = match.replace(' ', '')
        if clean_number.isdigit():
            amount = f"{clean_number} тыс. руб."
            if amount not in found_amounts:
                found_amounts.append(amount)

    # Паттерн для миллионов рублей
    pattern_millions = r'(\d{1,3}(?:\s?\d{3})*)\s+млн\s+руб\.'
    matches_millions = re.findall(pattern_millions, text)
    for match in matches_millions:
        clean_number = match.replace(' ', '')
        if clean_number.isdigit():
            amount = f"{clean_number} млн руб."
            if amount not in found_amounts:
                found_amounts.append(amount)

    # Удаляем дубликаты, сохраняя порядок
    unique_amounts = []
    for amount in found_amounts:
        if amount not in unique_amounts:
            unique_amounts.append(amount)

    return unique_amounts

# Тестирование функции на примерах из сообщений
print("=" * 70)
print("ЗАДАНИЕ 3.1 - ИЗВЛЕЧЕНИЕ СУММ ИЗ ТЕКСТА")
print("=" * 70)

# Тестируем на первых 10 сообщениях
test_messages = valid_messages[:10] if valid_messages else messages[:10]

print("\nТестирование на сообщениях:")
print("-" * 70)

for i, msg in enumerate(test_messages):
    text = msg.get('msg_text', '')
    if not text and 'msg_text' in msg:
        text = msg['msg_text']

    amounts = extract_amounts(text)

    print(f"\nСообщение {i+1}:")
    print(f"  Текст: {text[:80]}..." if len(text) > 80 else f"  Текст: {text}")
    print(f"  Найденные суммы: {amounts if amounts else 'не найдено'}")

# Дополнительное тестирование на различных форматах
print("\n" + "=" * 70)
print("Дополнительное тестирование на разных форматах:")
print("-" * 70)

test_texts = [
    "Долг составляет 15 000 000 руб.",
    "Сумма требований: 8 500 тыс. руб.",
    "Общая сумма задолженности: 42 млн руб.",
    "Сумма: 3 200 000 руб.",
    "Долг 15 000 000 руб. и пени 500 000 руб.",
    "Сумма соглашения: 6 800 000 руб.",
    "Имущество продано за 18 500 тыс. руб.",
    "Сумма ответственности: 27 000 000 руб.",
    "Долг: 56 000 000 руб.",
    "Сумма: 3 400 000 руб.",
]

for test_text in test_texts:
    amounts = extract_amounts(test_text)
    print(f"\nТекст: {test_text}")
    print(f"Результат: {amounts}")

ЗАДАНИЕ 3.1 - ИЗВЛЕЧЕНИЕ СУММ ИЗ ТЕКСТА

Тестирование на сообщениях:
----------------------------------------------------------------------

Сообщение 1:
  Текст: Сообщение о введении процедуры наблюдения в отношении ООО "Альфа-Строй". Арбитра...
  Найденные суммы: ['000 руб.', '15000000 руб.']

Сообщение 2:
  Текст: Уведомление о проведении собрания кредиторов ЗАО "Бета-Инвест". Сумма требований...
  Найденные суммы: ['500 тыс. руб.', '8500 тыс. руб.']

Сообщение 3:
  Текст: Сообщение о завершении конкурсного производства. Арбитражный управляющий Петров ...
  Найденные суммы: ['42 млн руб.']

Сообщение 4:
  Текст: Сообщение о признании должника банкротом. ОАО "Дельта-Логистик" признано банкрот...
  Найденные суммы: ['000 руб.', '3200000 руб.']

Сообщение 5:
  Текст: Уведомление о продаже имущества должника. ООО "Омега-Финанс". Начальная цена: 12...
  Найденные суммы: ['500 тыс. руб.', '12500 тыс. руб.']

Сообщение 6:
  Текст: Сообщение о введении внешнего управления. ПАО "Сигма-Энерго

### Задание 3.2 — Поиск упоминаний арбитражных судов

Напишите функцию `find_court_mentions(text)`, которая проверяет,
содержит ли текст упоминание арбитражного суда.

Ищите подстроку `"АС "` (с пробелом) в тексте через оператор `in`.
Верните `True` / `False`.

In [32]:
def find_court_mentions(text):
    """
    Проверяет, содержит ли текст упоминание арбитражного суда.
    Ищет подстроку "АС " (с пробелом) в тексте.
    Возвращает True или False.
    """
    if text is None or text == "":
        return False

    text = str(text)

    # Ищем "АС " (с пробелом после)
    if "АС " in text:
        return True

    # Также ищем "АС" в начале строки или с другими разделителями
    # на случай если пробел не стоит (например "АС г. Москвы")
    words = text.split()
    for word in words:
        if word == "АС" or word.startswith("АС"):
            return True

    return False

# Тестирование функции
print("=" * 70)
print("ЗАДАНИЕ 3.2 - ПОИСК УПОМИНАНИЙ АРБИТРАЖНЫХ СУДОВ")
print("=" * 70)

# Тестируем на первых 15 валидных сообщениях
test_messages = valid_messages[:15] if valid_messages else messages[:15]

print("\nРезультаты проверки сообщений:")
print("-" * 70)

for i, msg in enumerate(test_messages):
    text = msg.get('msg_text', '')
    if not text and 'msg_text' in msg:
        text = msg['msg_text']

    has_court = find_court_mentions(text)

    # Сокращаем текст для вывода
    short_text = text[:60] + "..." if len(text) > 60 else text

    print(f"\n{i+1}. Текст: {short_text}")
    print(f"   Содержит 'АС': {'✅ ДА' if has_court else '❌ НЕТ'}")

# Подсчет статистики по всем валидным сообщениям
print("\n" + "=" * 70)
print("СТАТИСТИКА ПО ВСЕМ ВАЛИДНЫМ СООБЩЕНИЯМ:")
print("-" * 70)

if valid_messages:
    court_mentions_count = 0
    for msg in valid_messages:
        if find_court_mentions(msg['msg_text']):
            court_mentions_count += 1

    print(f"Всего валидных сообщений: {len(valid_messages)}")
    print(f"Сообщений с упоминанием 'АС': {court_mentions_count}")
    print(f"Сообщений без упоминания 'АС': {len(valid_messages) - court_mentions_count}")
    print(f"Процент сообщений с упоминанием суда: {court_mentions_count / len(valid_messages) * 100:.1f}%")
else:
    print("Нет валидных сообщений для анализа")

# Дополнительное тестирование на разных примерах
print("\n" + "=" * 70)
print("ДОПОЛНИТЕЛЬНОЕ ТЕСТИРОВАНИЕ:")
print("-" * 70)

test_examples = [
    "АС Свердловской области.",
    "АС г. Москвы",
    "Арбитражный суд г. Санкт-Петербурга",
    "АС Краснодарского края",
    "АС Московской области",
    "АС Челябинской области",
    "АС Владимирской области",
    "АС Тверской области",
    "АС Ростовской области",
    "Текст без упоминания суда",
    "АС ",
    "АС в начале строки",
]

for example in test_examples:
    result = find_court_mentions(example)
    print(f"'{example}' -> {result}")

ЗАДАНИЕ 3.2 - ПОИСК УПОМИНАНИЙ АРБИТРАЖНЫХ СУДОВ

Результаты проверки сообщений:
----------------------------------------------------------------------

1. Текст: Сообщение о введении процедуры наблюдения в отношении ООО "А...
   Содержит 'АС': ✅ ДА

2. Текст: Уведомление о проведении собрания кредиторов ЗАО "Бета-Инвес...
   Содержит 'АС': ✅ ДА

3. Текст: Сообщение о завершении конкурсного производства. Арбитражный...
   Содержит 'АС': ❌ НЕТ

4. Текст: Сообщение о признании должника банкротом. ОАО "Дельта-Логист...
   Содержит 'АС': ✅ ДА

5. Текст: Уведомление о продаже имущества должника. ООО "Омега-Финанс"...
   Содержит 'АС': ✅ ДА

6. Текст: Сообщение о введении внешнего управления. ПАО "Сигма-Энерго"...
   Содержит 'АС': ✅ ДА

7. Текст: Информация о требовании кредитора. Сумма требования: 950 000...
   Содержит 'АС': ✅ ДА

8. Текст: Сообщение об оспаривании сделки должника. ООО "Тета-Консалти...
   Содержит 'АС': ✅ ДА

9. Текст: Сообщение о подаче заявления о банкротстве. Долг сос

### Задание 3.3 — Извлечение ФИО арбитражных управляющих

Напишите функцию `extract_manager_name(text)`, которая ищет ФИО арбитражного управляющего.

Алгоритм:
1. Проверьте, содержит ли текст подстроку `"арбитражный управляющий"`.
2. Если да — найдите позицию этой подстроки, возьмите текст после неё.
3. С помощью `.split()` извлеките следующие 3 слова (ФИО).
4. Соедините через пробел и верните.
5. Если не найдено — верните `None`.

In [33]:
def extract_manager_name(text):
    """
    Ищет ФИО арбитражного управляющего в тексте.
    Возвращает строку с ФИО или None, если не найдено.
    """
    if text is None or text == "":
        return None

    text = str(text)

    # Проверяем, содержит ли текст подстроку "арбитражный управляющий"
    if "арбитражный управляющий" not in text.lower():
        return None

    # Находим позицию подстроки
    keyword = "арбитражный управляющий"
    pos = text.lower().find(keyword)

    # Берем текст после ключевой фразы
    after_keyword = text[pos + len(keyword):]

    # Разбиваем на слова
    words = after_keyword.split()

    # Ищем первые 3 слова (ФИО)
    # Пропускаем возможные точки и запятые в начале
    name_parts = []
    for word in words:
        # Очищаем слово от знаков препинания
        cleaned_word = word.strip('.,;:!?')
        if cleaned_word and len(cleaned_word) > 1:  # Берем только непустые слова
            name_parts.append(cleaned_word)
            if len(name_parts) == 3:
                break

    # Если нашли 3 слова, соединяем их
    if len(name_parts) == 3:
        return " ".join(name_parts)

    # Если нашли меньше 3 слов, возвращаем то, что нашли
    elif len(name_parts) > 0:
        return " ".join(name_parts)

    return None

# Тестирование функции
print("=" * 70)
print("ЗАДАНИЕ 3.3 - ИЗВЛЕЧЕНИЕ ФИО АРБИТРАЖНЫХ УПРАВЛЯЮЩИХ")
print("=" * 70)

# Тестируем на примерах из сообщений
print("\nРезультаты поиска ФИО управляющих:")
print("-" * 70)

# Собираем сообщения, которые содержат упоминание управляющего
manager_messages = []
for msg in valid_messages:
    if "арбитражный управляющий" in msg['msg_text'].lower():
        manager_messages.append(msg)

if manager_messages:
    for i, msg in enumerate(manager_messages[:10]):  # Показываем первые 10
        text = msg['msg_text']
        manager_name = extract_manager_name(text)

        # Сокращаем текст для вывода
        short_text = text[:80] + "..." if len(text) > 80 else text

        print(f"\n{i+1}. Текст: {short_text}")
        print(f"   ФИО управляющего: {manager_name if manager_name else 'не найдено'}")
else:
    print("Нет сообщений с упоминанием арбитражного управляющего")

# Дополнительное тестирование на разных форматах
print("\n" + "=" * 70)
print("ДОПОЛНИТЕЛЬНОЕ ТЕСТИРОВАНИЕ НА РАЗНЫХ ФОРМАТАХ:")
print("-" * 70)

test_texts = [
    "Арбитражный управляющий Иванов Иван Иванович.",
    "арбитражный управляющий Петров Пётр Петрович",
    "Арбитражный управляющий Сидоров Сергей Сергеевич. Сумма долга",
    "Арбитражный управляющий Фёдоров Фёдор Фёдорович, АС г. Москвы",
    "Арбитражный управляющий Николаев Николай Николаевич: сообщение",
    "Арбитражный управляющий Алексеев Алексей Алексеевич",
    "Арбитражный управляющий Дмитриев Дмитрий Дмитриевич - назначен",
    "Арбитражный управляющий Сергеев Сергей Сергеевич",
    "Арбитражный управляющий Андреев Андрей Андреевич",
    "Арбитражный управляющий Васильев Василий Васильевич",
    "Текст без упоминания управляющего",
    "Арбитражный управляющий",
    "арбитражный управляющий Иван Петров",  # Только 2 слова
]

for test_text in test_texts:
    manager_name = extract_manager_name(test_text)
    print(f"\nТекст: {test_text}")
    print(f"Результат: {manager_name}")

# Подсчет статистики по всем валидным сообщениям
print("\n" + "=" * 70)
print("СТАТИСТИКА ПО ВСЕМ ВАЛИДНЫМ СООБЩЕНИЯМ:")
print("-" * 70)

if valid_messages:
    total_with_manager = 0
    successful_extractions = 0

    for msg in valid_messages:
        if "арбитражный управляющий" in msg['msg_text'].lower():
            total_with_manager += 1
            manager_name = extract_manager_name(msg['msg_text'])
            if manager_name:
                successful_extractions += 1

    print(f"Всего валидных сообщений: {len(valid_messages)}")
    print(f"Сообщений с упоминанием 'арбитражный управляющий': {total_with_manager}")
    print(f"Успешно извлечено ФИО: {successful_extractions}")

    if total_with_manager > 0:
        print(f"Процент успешного извлечения: {successful_extractions / total_with_manager * 100:.1f}%")
    else:
        print("Процент успешного извлечения: 0%")
else:
    print("Нет валидных сообщений для анализа")

ЗАДАНИЕ 3.3 - ИЗВЛЕЧЕНИЕ ФИО АРБИТРАЖНЫХ УПРАВЛЯЮЩИХ

Результаты поиска ФИО управляющих:
----------------------------------------------------------------------

1. Текст: Сообщение о введении процедуры наблюдения в отношении ООО "Альфа-Строй". Арбитра...
   ФИО управляющего: Иванов Иван Иванович

2. Текст: Сообщение о завершении конкурсного производства. Арбитражный управляющий Петров ...
   ФИО управляющего: Петров Пётр Петрович

3. Текст: Сообщение о введении внешнего управления. ПАО "Сигма-Энерго". Арбитражный управл...
   ФИО управляющего: Сидоров Сергей Сергеевич

4. Текст: Уведомление о проведении торгов. ООО "Каппа-Девелопмент". Начальная цена имущест...
   ФИО управляющего: Николаев Николай Николаевич

5. Текст: Сообщение о проведении собрания кредиторов. ООО "Кси-Ритейл". Сумма требований: ...
   ФИО управляющего: Фёдоров Фёдор Фёдорович

6. Текст: Сообщение о введении процедуры финансового оздоровления. ПАО "Тау-Банк". Сумма д...
   ФИО управляющего: Алексеев Алексей Алексеев

### Задание 3.4 — Обогащение данных

Для каждого **валидного** сообщения добавьте поля:
- `amounts` — список извлечённых сумм (функция `extract_amounts`)
- `has_court_mention` — `True`/`False` (функция `find_court_mentions`)
- `manager_name` — ФИО или `None` (функция `extract_manager_name`)

In [34]:
enriched_messages = []

for msg in valid_messages:
    enriched_msg = msg.copy()  # Создаем копию, чтобы не изменять оригинал

    # Добавляем поле amounts
    enriched_msg['amounts'] = extract_amounts(msg['msg_text'])

    # Добавляем поле has_court_mention
    enriched_msg['has_court_mention'] = find_court_mentions(msg['msg_text'])

    # Добавляем поле manager_name
    enriched_msg['manager_name'] = extract_manager_name(msg['msg_text'])

    enriched_messages.append(enriched_msg)

print(f"\n✅ Обогащено {len(enriched_messages)} сообщений")
print(f"❌ Пропущено {len(error_records)} невалидных сообщений")

# Вывод примера обогащенных данных
print("\n" + "=" * 70)
print("ПРИМЕРЫ ОБОГАЩЕННЫХ СООБЩЕНИЙ:")
print("=" * 70)

# Показываем первые 5 обогащенных сообщений
for i, msg in enumerate(enriched_messages[:5]):
    print(f"\n--- Сообщение {i+1} ---")
    print(f"Номер дела: {msg['case_number']}")
    print(f"Тип: {msg['type']}")
    print(f"Дата: {msg['date_published'].strftime('%d.%m.%Y') if msg['date_published'] else 'None'}")
    print(f"Организация: {msg.get('org_name', 'Не указана')}")
    print(f"Регион: {msg.get('region', 'Не указан')}")
    print(f"Текст: {msg['msg_text'][:100]}..." if len(msg['msg_text']) > 100 else f"Текст: {msg['msg_text']}")
    print(f"Суммы: {msg['amounts'] if msg['amounts'] else 'не найдены'}")
    print(f"Упоминание суда: {'Да' if msg['has_court_mention'] else 'Нет'}")
    print(f"Арбитражный управляющий: {msg['manager_name'] if msg['manager_name'] else 'не указан'}")

# Статистика по обогащенным данным
print("\n" + "=" * 70)
print("СТАТИСТИКА ПО ОБОГАЩЕННЫМ ДАННЫМ:")
print("=" * 70)

if enriched_messages:
    # Подсчет сообщений с суммами
    messages_with_amounts = sum(1 for msg in enriched_messages if msg['amounts'])

    # Подсчет сообщений с упоминанием суда
    messages_with_court = sum(1 for msg in enriched_messages if msg['has_court_mention'])

    # Подсчет сообщений с управляющим
    messages_with_manager = sum(1 for msg in enriched_messages if msg['manager_name'] is not None)

    # Подсчет общего количества найденных сумм
    total_amounts = sum(len(msg['amounts']) for msg in enriched_messages)

    print(f"\n📊 Общая статистика:")
    print(f"  Всего обогащенных сообщений: {len(enriched_messages)}")
    print(f"  Сообщения с найденными суммами: {messages_with_amounts} ({messages_with_amounts/len(enriched_messages)*100:.1f}%)")
    print(f"  Сообщения с упоминанием суда: {messages_with_court} ({messages_with_court/len(enriched_messages)*100:.1f}%)")
    print(f"  Сообщения с указанием управляющего: {messages_with_manager} ({messages_with_manager/len(enriched_messages)*100:.1f}%)")
    print(f"  Всего найдено сумм: {total_amounts}")

    # Статистика по типам сообщений
    print(f"\n📋 Статистика по типам сообщений:")
    type_stats = defaultdict(int)
    for msg in enriched_messages:
        type_stats[msg['type']] += 1

    for msg_type, count in sorted(type_stats.items(), key=lambda x: x[1], reverse=True):
        print(f"  {msg_type}: {count}")

    # Статистика по регионам
    print(f"\n🗺️ Статистика по регионам (топ-5):")
    region_stats = defaultdict(int)
    for msg in enriched_messages:
        if msg.get('region'):
            region_stats[msg['region']] += 1

    for region, count in sorted(region_stats.items(), key=lambda x: x[1], reverse=True)[:5]:
        print(f"  {region}: {count}")

    # Сообщения с наибольшим количеством сумм
    print(f"\n💰 Сообщения с наибольшим количеством найденных сумм:")
    sorted_by_amounts = sorted(enriched_messages, key=lambda x: len(x['amounts']), reverse=True)
    for msg in sorted_by_amounts[:3]:
        if msg['amounts']:
            print(f"  Дело {msg['case_number']}: найдено {len(msg['amounts'])} сумм(а) -> {msg['amounts']}")

# Сохраняем результат в переменную enriched_messages для дальнейшего использования
print("\n" + "=" * 70)
print("✅ Обогащенные данные сохранены в переменную 'enriched_messages'")
print(f"   Тип: list из {len(enriched_messages)} словарей")


✅ Обогащено 48 сообщений
❌ Пропущено 6 невалидных сообщений

ПРИМЕРЫ ОБОГАЩЕННЫХ СООБЩЕНИЙ:

--- Сообщение 1 ---
Номер дела: А60-12345/2025
Тип: Введение процедуры
Дата: 15.01.2025
Организация: ООО "Альфа-Строй"
Регион: Москва
Текст: Сообщение о введении процедуры наблюдения в отношении ООО "Альфа-Строй". Арбитражный управляющий Ива...
Суммы: ['000 руб.', '15000000 руб.']
Упоминание суда: Да
Арбитражный управляющий: Иванов Иван Иванович

--- Сообщение 2 ---
Номер дела: А60-56789/2025
Тип: Собрание кредиторов
Дата: 10.02.2025
Организация: ЗАО "Бета-Инвест"
Регион: Москва
Текст: Уведомление о проведении собрания кредиторов ЗАО "Бета-Инвест". Сумма требований: 8 500 тыс. руб. АС...
Суммы: ['500 тыс. руб.', '8500 тыс. руб.']
Упоминание суда: Да
Арбитражный управляющий: не указан

--- Сообщение 3 ---
Номер дела: А60-11111/2025
Тип: Завершение процедуры
Дата: 15.03.2025
Организация: ООО "Гамма-Трейд"
Регион: Свердловская область
Текст: Сообщение о завершении конкурсного производства. Арбитр

### Задание 3.5 — Аналитика

Постройте следующие показатели по **валидным** сообщениям:

1. **Количество сообщений по типам** — словарь `{тип: количество}`
2. **Количество сообщений по регионам** — словарь `{регион: количество}`, пропуская `None`
3. **Топ-5 публикаторов** по количеству сообщений — `sorted(key=lambda...)`
4. **Таймлайн** — сообщения, отсортированные по дате публикации

In [35]:
print("\n1️⃣ КОЛИЧЕСТВО СООБЩЕНИЙ ПО ТИПАМ:")
print("-" * 50)

type_stats = defaultdict(int)
for msg in enriched_messages:
    type_stats[msg['type']] += 1

for msg_type, count in sorted(type_stats.items(), key=lambda x: x[1], reverse=True):
    print(f"   {msg_type}: {count}")

# 2. Количество сообщений по регионам (пропуская None)
print("\n2️⃣ КОЛИЧЕСТВО СООБЩЕНИЙ ПО РЕГИОНАМ:")
print("-" * 50)

region_stats = defaultdict(int)
for msg in enriched_messages:
    if msg.get('region') is not None:
        region_stats[msg['region']] += 1

for region, count in sorted(region_stats.items(), key=lambda x: x[1], reverse=True):
    print(f"   {region}: {count}")

# 3. Топ-5 публикаторов по количеству сообщений
print("\n3️⃣ ТОП-5 ПУБЛИКАТОРОВ ПО КОЛИЧЕСТВУ СООБЩЕНИЙ:")
print("-" * 50)

publisher_stats = defaultdict(int)
for msg in enriched_messages:
    publisher_stats[msg['publisher_inn']] += 1

# Сортируем по количеству сообщений (по убыванию) и берем топ-5
top_publishers = sorted(publisher_stats.items(), key=lambda x: x[1], reverse=True)[:5]

for i, (publisher_inn, count) in enumerate(top_publishers, 1):
    # Пытаемся найти название организации по ИНН
    org_name = org_by_inn.get(publisher_inn, {}).get('name', 'Неизвестная организация')
    print(f"   {i}. ИНН {publisher_inn} ({org_name[:40]}): {count} сообщений")

# 4. Таймлайн — сообщения, отсортированные по дате публикации
print("\n4️⃣ ТАЙМЛАЙН СООБЩЕНИЙ (по дате публикации):")
print("-" * 50)

# Сортируем сообщения по дате публикации
sorted_by_date = sorted(enriched_messages, key=lambda x: x['date_published'])

print(f"\n   Всего сообщений в таймлайне: {len(sorted_by_date)}")
print(f"   Период: с {sorted_by_date[0]['date_published'].strftime('%d.%m.%Y')} по {sorted_by_date[-1]['date_published'].strftime('%d.%m.%Y')}")
print("\n   Детальный таймлайн:")
print("   " + "-" * 55)

for msg in sorted_by_date:
    date_str = msg['date_published'].strftime('%d.%m.%Y')
    # Сокращаем текст для красивого вывода
    short_text = msg['msg_text'][:50] + "..." if len(msg['msg_text']) > 50 else msg['msg_text']
    print(f"   {date_str} | {msg['type']:25} | {msg['case_number']:15} | {short_text}")

# Дополнительная аналитика
print("\n" + "=" * 70)
print("ДОПОЛНИТЕЛЬНАЯ АНАЛИТИКА:")
print("=" * 70)

# Распределение по месяцам
print("\n📅 РАСПРЕДЕЛЕНИЕ ПО МЕСЯЦАМ:")
month_stats = defaultdict(int)
for msg in sorted_by_date:
    month_key = msg['date_published'].strftime('%Y-%m')
    month_stats[month_key] += 1

for month, count in sorted(month_stats.items()):
    print(f"   {month}: {count} сообщений")

# Сообщения с управляющими по типам
print("\n👨‍⚖️ СООБЩЕНИЯ С УПОМИНАНИЕМ АРБИТРАЖНЫХ УПРАВЛЯЮЩИХ ПО ТИПАМ:")
manager_by_type = defaultdict(int)
for msg in enriched_messages:
    if msg['manager_name']:
        manager_by_type[msg['type']] += 1

for msg_type, count in sorted(manager_by_type.items(), key=lambda x: x[1], reverse=True):
    total_of_type = type_stats[msg_type]
    percentage = (count / total_of_type * 100) if total_of_type > 0 else 0
    print(f"   {msg_type}: {count} из {total_of_type} ({percentage:.1f}%)")

# Сообщения с упоминанием суда по регионам
print("\n🏛️ СООБЩЕНИЯ С УПОМИНАНИЕМ СУДА ПО РЕГИОНАМ (топ-5):")
court_by_region = defaultdict(int)
court_total_by_region = defaultdict(int)

for msg in enriched_messages:
    if msg.get('region'):
        court_total_by_region[msg['region']] += 1
        if msg['has_court_mention']:
            court_by_region[msg['region']] += 1

# Сортируем по количеству сообщений с упоминанием суда
top_regions = sorted(court_by_region.items(), key=lambda x: x[1], reverse=True)[:5]

for region, count in top_regions:
    total = court_total_by_region[region]
    percentage = (count / total * 100) if total > 0 else 0
    print(f"   {region}: {count} из {total} ({percentage:.1f}%)")

# Вывод итоговой сводки
print("\n" + "=" * 70)
print("ИТОГОВАЯ СВОДКА:")
print("=" * 70)
print(f"✅ Всего валидных сообщений: {len(enriched_messages)}")
print(f"📊 Уникальных типов сообщений: {len(type_stats)}")
print(f"🗺️ Уникальных регионов: {len([r for r in region_stats if r is not None])}")
print(f"👥 Уникальных публикаторов: {len(publisher_stats)}")
print(f"📅 Самый ранний таймлайн: {sorted_by_date[0]['date_published'].strftime('%d.%m.%Y')}")
print(f"📅 Самый поздний таймлайн: {sorted_by_date[-1]['date_published'].strftime('%d.%m.%Y')}")


1️⃣ КОЛИЧЕСТВО СООБЩЕНИЙ ПО ТИПАМ:
--------------------------------------------------
   Введение процедуры: 8
   Завершение процедуры: 6
   Продажа имущества: 6
   Требование кредитора: 4
   Собрание кредиторов: 3
   Признание банкротом: 3
   Оспаривание сделки: 3
   Подача заявления: 3
   Мировое соглашение: 3
   Передача дела: 3
   Субсидиарная ответственность: 2
   Назначение управляющего: 2
   Отстранение управляющего: 1
   Жалоба на управляющего: 1

2️⃣ КОЛИЧЕСТВО СООБЩЕНИЙ ПО РЕГИОНАМ:
--------------------------------------------------
   Москва: 25
   Свердловская область: 6
   Санкт-Петербург: 5
   Краснодарский край: 3
   Челябинская область: 3
   Ростовская область: 2
   Московская область: 2
   Владимирская область: 1

3️⃣ ТОП-5 ПУБЛИКАТОРОВ ПО КОЛИЧЕСТВУ СООБЩЕНИЙ:
--------------------------------------------------
   1. ИНН 7701234567 (ООО "Альфа-Строй"): 3 сообщений
   2. ИНН 7706567890 (ООО "Тета-Консалтинг"): 3 сообщений
   3. ИНН 7702345678 (ЗАО "Бета-Инвест"): 2 соо

---
# 🟣 Отчётность (0-1 балл)
---

### Задание 4.1 — Подготовка данных для сохранения

Подготовьте данные для записи в JSON. Даты нужно преобразовать обратно в строки (`strftime`),
чтобы JSON был читаемым.

In [37]:
from datetime import datetime
def prepare_for_json(messages_list):
    """
    Преобразует список сообщений в JSON-совместимый формат.
    Дата преобразуется в строку, чтобы JSON был читаемым.
    """
    prepared_messages = []

    for msg in messages_list:
        # Создаем копию сообщения
        prepared_msg = msg.copy()

        # Преобразуем datetime в строку, если это datetime объект
        if 'date_published' in prepared_msg and isinstance(prepared_msg['date_published'], datetime):
            prepared_msg['date_published'] = prepared_msg['date_published'].strftime('%Y-%m-%d %H:%M:%S')

        # Преобразуем другие возможные datetime объекты (если есть)
        for key, value in prepared_msg.items():
            if isinstance(value, datetime):
                prepared_msg[key] = value.strftime('%Y-%m-%d %H:%M:%S')

        prepared_messages.append(prepared_msg)

    return prepared_messages

# Подготавливаем данные
json_ready_messages = prepare_for_json(enriched_messages)

print(f"\n✅ Подготовлено {len(json_ready_messages)} сообщений для сохранения в JSON")
print(f"\nПример подготовленного сообщения (первое):")
print("-" * 70)

if json_ready_messages:
    example = json_ready_messages[0]
    for key, value in example.items():
        # Ограничиваем длину текста для красивого вывода
        if key == 'msg_text' and len(str(value)) > 100:
            print(f"  {key}: {str(value)[:100]}...")
        elif key == 'amounts' and value:
            print(f"  {key}: {value}")
        else:
            print(f"  {key}: {value}")

# Сохраняем в JSON файл
output_file = Path("/content/final_hw_data/enriched_bankruptcy_messages.json")

with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(json_ready_messages, f, ensure_ascii=False, indent=2)

print(f"\n💾 Данные сохранены в файл: {output_file.absolute()}")
print(f"   Размер файла: {output_file.stat().st_size} байт")


✅ Подготовлено 48 сообщений для сохранения в JSON

Пример подготовленного сообщения (первое):
----------------------------------------------------------------------
  publisher_inn: 7701234567
  msg_text: Сообщение о введении процедуры наблюдения в отношении ООО "Альфа-Строй". Арбитражный управляющий Ива...
  date_published: 2025-01-15 00:00:00
  type: Введение процедуры
  case_number: А60-12345/2025
  org_name: ООО "Альфа-Строй"
  region: Москва
  amounts: ['000 руб.', '15000000 руб.']
  has_court_mention: True
  manager_name: Иванов Иван Иванович

💾 Данные сохранены в файл: /content/final_hw_data/enriched_bankruptcy_messages.json
   Размер файла: 29796 байт


### Задание 4.2 — Сохранение `analysis_results.json`

Сохраните файл `analysis_results.json` со следующей структурой:
```json
{
  "valid_messages": [...],
  "type_stats": {...},
  "region_stats": {...},
  "priority_messages": [...]
}
```

In [38]:
def convert_datetime_to_str(obj):
    """Рекурсивно преобразует datetime объекты в строки"""
    if isinstance(obj, datetime):
        return obj.strftime('%Y-%m-%d %H:%M:%S')
    elif isinstance(obj, dict):
        return {key: convert_datetime_to_str(value) for key, value in obj.items()}
    elif isinstance(obj, list):
        return [convert_datetime_to_str(item) for item in obj]
    else:
        return obj

# 1. Собираем статистику по типам
type_stats = defaultdict(int)
for msg in enriched_messages:
    type_stats[msg['type']] += 1

# 2. Собираем статистику по регионам (пропуская None)
region_stats = defaultdict(int)
for msg in enriched_messages:
    if msg.get('region') is not None:
        region_stats[msg['region']] += 1

# 3. Находим приоритетные сообщения
priority_set = set(priority_cases)
priority_messages = []

for msg in enriched_messages:
    if msg['case_number'] in priority_set:
        priority_messages.append(msg)

# Подготавливаем данные для сохранения
# Преобразуем валидные сообщения в JSON-совместимый формат
valid_messages_for_json = convert_datetime_to_str(enriched_messages)

# Преобразуем приоритетные сообщения
priority_messages_for_json = convert_datetime_to_str(priority_messages)

# Создаем структуру для сохранения
analysis_results = {
    "valid_messages": valid_messages_for_json,
    "type_stats": dict(type_stats),
    "region_stats": dict(region_stats),
    "priority_messages": priority_messages_for_json
}

# Сохраняем в файл
output_file_2 = Path("/content/final_hw_data/analysis_results.json")

with open(output_file_2, 'w', encoding='utf-8') as f:
    json.dump(analysis_results, f, ensure_ascii=False, indent=2)

print(f"\n✅ Файл сохранен: {output_file_2.absolute()}")
print(f"   Размер файла: {output_file_2.stat().st_size} байт")


✅ Файл сохранен: /content/final_hw_data/analysis_results.json
   Размер файла: 53516 байт


### Задание 4.3 — Сохранение `validation_errors.json`

Сохраните проблемные записи с описанием ошибок.

In [39]:
def convert_datetime_to_str(obj):
    """Рекурсивно преобразует datetime объекты в строки"""
    if isinstance(obj, datetime):
        return obj.strftime('%Y-%m-%d %H:%M:%S')
    elif isinstance(obj, dict):
        return {key: convert_datetime_to_str(value) for key, value in obj.items()}
    elif isinstance(obj, list):
        return [convert_datetime_to_str(item) for item in obj]
    else:
        return obj

# Подготавливаем данные об ошибках для сохранения
validation_errors = []

for error_record in error_records:
    # Преобразуем оригинальное сообщение в JSON-совместимый формат
    original_msg = error_record['original_message'].copy()

    # Если в сообщении есть datetime объекты, преобразуем их в строки
    if 'date_published' in original_msg and isinstance(original_msg['date_published'], datetime):
        original_msg['date_published'] = original_msg['date_published'].strftime('%Y-%m-%d %H:%M:%S')

    # Создаем запись об ошибке
    error_entry = {
        "index": error_record['index'],
        "original_message": original_msg,
        "errors": error_record['errors'],
        "error_count": len(error_record['errors'])
    }

    validation_errors.append(error_entry)

# Добавляем общую статистику ошибок
error_summary = {
    "total_errors_count": len(validation_errors),
    "error_types": dict(error_stats),
    "validation_errors": validation_errors
}

# Сохраняем в файл
output_file_3 = Path("/content/final_hw_data/validation_errors.json")

with open(output_file_3, 'w', encoding='utf-8') as f:
    json.dump(error_summary, f, ensure_ascii=False, indent=2)

print(f"\n✅ Файл сохранен: {output_file.absolute()}")
print(f"   Размер файла: {output_file.stat().st_size} байт")



✅ Файл сохранен: /content/final_hw_data/enriched_bankruptcy_messages.json
   Размер файла: 29796 байт


### Задание 4.4 — Текстовый отчёт `summary_report.txt`

Сформируйте текстовый аналитический отчёт для руководства с помощью **f-строк**.

Отчёт должен содержать:
- Заголовок и дату
- Общую статистику (всего сообщений, валидных, ошибочных)
- Статистику по типам сообщений
- Статистику по регионам
- Топ-5 публикаторов
- Список приоритетных дел
- Список найденных арбитражных управляющих

In [43]:
total_messages = len(messages)
valid_count = len(valid_messages)
error_count = len(error_records)

# 2. Статистика по типам сообщений
type_stats = defaultdict(int)
for msg in valid_messages:
    type_stats[msg['type']] += 1

# 3. Статистика по регионам
region_stats = defaultdict(int)
for msg in valid_messages:
    if msg.get('region'):
        region_stats[msg['region']] += 1

# 4. Топ-5 публикаторов
publisher_stats = defaultdict(int)
for msg in valid_messages:
    publisher_stats[msg['publisher_inn']] += 1

top_publishers = sorted(publisher_stats.items(), key=lambda x: x[1], reverse=True)[:5]

# 5. Приоритетные дела
priority_set = set(priority_cases)
priority_messages = [msg for msg in valid_messages if msg['case_number'] in priority_set]
priority_list = sorted(set(msg['case_number'] for msg in priority_messages))

# 6. Список арбитражных управляющих
managers = set()
for msg in valid_messages:
    if "арбитражный управляющий" in msg['msg_text'].lower():
        # Простое извлечение: ищем фразу и берем следующие слова
        text = msg['msg_text']
        keyword = "арбитражный управляющий"
        pos = text.lower().find(keyword)
        if pos != -1:
            after = text[pos + len(keyword):]
            words = after.split()
            name_parts = []
            for word in words:
                cleaned = word.strip('.,;:!?')
                if cleaned and len(cleaned) > 1 and not cleaned.isdigit():
                    name_parts.append(cleaned)
                    if len(name_parts) == 3:
                        break
            if len(name_parts) >= 2:
                managers.add(" ".join(name_parts[:3]))

# Формируем отчёт с помощью f-строк
report = f"""
{'='*80}
         АНАЛИТИЧЕСКИЙ ОТЧЁТ ПО БАНКРОТСТВАМ
         Дата формирования: {datetime.now().strftime('%d.%m.%Y %H:%M:%S')}
{'='*80}

1. ОБЩАЯ СТАТИСТИКА
{'-'*50}
   Всего сообщений в системе:     {total_messages}
   Валидных сообщений:            {valid_count} ({valid_count/total_messages*100:.1f}%)
   Ошибочных сообщений:           {error_count} ({error_count/total_messages*100:.1f}%)
   Уникальных номеров дел:        {len(set(msg['case_number'] for msg in valid_messages))}
   Уникальных публикаторов:       {len(publisher_stats)}

2. СТАТИСТИКА ПО ТИПАМ СООБЩЕНИЙ
{'-'*50}
"""
for msg_type, count in sorted(type_stats.items(), key=lambda x: x[1], reverse=True):
    report += f"   {msg_type:<35} {count:>3} ({count/valid_count*100:>5.1f}%)\n"

report += f"""
3. СТАТИСТИКА ПО РЕГИОНАМ (ТОП-10)
{'-'*50}
"""
for region, count in sorted(region_stats.items(), key=lambda x: x[1], reverse=True)[:10]:
    report += f"   {region:<40} {count:>3} ({count/valid_count*100:>5.1f}%)\n"

if not region_stats:
    report += "   Данные по регионам отсутствуют\n"

report += f"""
4. ТОП-5 ПУБЛИКАТОРОВ ПО КОЛИЧЕСТВУ СООБЩЕНИЙ
{'-'*50}
"""
for i, (inn, count) in enumerate(top_publishers, 1):
    org_name = org_by_inn.get(inn, {}).get('name', 'Неизвестная организация')
    report += f"   {i}. ИНН {inn}\n"
    report += f"      {org_name[:50]}\n"
    report += f"      Сообщений: {count}\n\n"

report += f"""
5. ПРИОРИТЕТНЫЕ ДЕЛА
{'-'*50}
   Всего приоритетных дел в списке:      {len(priority_set)}
   Из них найдено в сообщениях:          {len(priority_list)}
   Процент совпадения:                   {len(priority_list)/len(priority_set)*100:.1f}%

   Список приоритетных дел, присутствующих в сообщениях:
"""
for case in sorted(priority_list):
    # Находим тип сообщения по этому делу
    msg_type = next((msg['type'] for msg in priority_messages if msg['case_number'] == case), 'неизвестно')
    report += f"   • {case} — {msg_type}\n"

if not priority_list:
    report += "   Нет совпадающих дел\n"

report += f"""
6. СПИСОК НАЙДЕННЫХ АРБИТРАЖНЫХ УПРАВЛЯЮЩИХ
{'-'*50}
"""
if managers:
    for i, manager in enumerate(sorted(managers), 1):
        report += f"   {i:>2}. {manager}\n"
    report += f"\n   Всего уникальных управляющих: {len(managers)}\n"
else:
    report += "   Арбитражные управляющие не найдены\n"

report += f"""
7. ДОПОЛНИТЕЛЬНАЯ ИНФОРМАЦИЯ
{'-'*50}
   Сообщений с упоминанием суда:     {sum(1 for msg in valid_messages if 'АС' in msg['msg_text'] or 'АС ' in msg['msg_text'])}
   Сообщений с указанием сумм:       {sum(1 for msg in valid_messages if any(word in msg['msg_text'] for word in ['руб', 'млн', 'тыс.']))}
   Самый частый тип сообщения:       {max(type_stats.items(), key=lambda x: x[1])[0] if type_stats else 'нет данных'}
   Самый активный регион:            {max(region_stats.items(), key=lambda x: x[1])[0] if region_stats else 'нет данных'}

{'='*80}
              КОНЕЦ ОТЧЁТА
{'='*80}
"""

# Выводим отчёт на экран
print(report)

# Сохраняем отчёт в текстовый файл
report_file = Path('summary_report.txt')
with open(report_file, 'w', encoding='utf-8') as f:
    f.write(report)

print(f"\n✅ Отчёт сохранён в файл: {report_file.absolute()}")
print(f"   Размер файла: {report_file.stat().st_size} байт")


         АНАЛИТИЧЕСКИЙ ОТЧЁТ ПО БАНКРОТСТВАМ
         Дата формирования: 25.04.2026 14:06:51

1. ОБЩАЯ СТАТИСТИКА
--------------------------------------------------
   Всего сообщений в системе:     54
   Валидных сообщений:            48 (88.9%)
   Ошибочных сообщений:           6 (11.1%)
   Уникальных номеров дел:        16
   Уникальных публикаторов:       31

2. СТАТИСТИКА ПО ТИПАМ СООБЩЕНИЙ
--------------------------------------------------
   Введение процедуры                    8 ( 16.7%)
   Завершение процедуры                  6 ( 12.5%)
   Продажа имущества                     6 ( 12.5%)
   Требование кредитора                  4 (  8.3%)
   Собрание кредиторов                   3 (  6.2%)
   Признание банкротом                   3 (  6.2%)
   Оспаривание сделки                    3 (  6.2%)
   Подача заявления                      3 (  6.2%)
   Мировое соглашение                    3 (  6.2%)
   Передача дела                         3 (  6.2%)
   Субсидиарная ответственнос

---
## ✅ Итоги

Если вы корректно выполнили все 4 уровня, у вас должны получиться:

| Файл | Описание |
|------|----------|
| `analysis_results.json` | Валидные сообщения + статистика + приоритетные дела |
| `validation_errors.json` | Проблемные записи с описанием ошибок |
| `summary_report.txt` | Текстовый аналитический отчёт для руководства |

